In [1]:
!git clone https://github.com/zihenglyu/TOPSIS.git

Cloning into 'TOPSIS'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 14), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 16.12 KiB | 16.12 MiB/s, done.
Resolving deltas: 100% (14/14), done.


# TOPSIS Demo Notebook

This notebook shows how to use the `TOPSIS` class from [TOPSIS.py](TOPSIS.py) and explains the full workflow with formulas and a worked example.

## What TOPSIS does

TOPSIS ranks alternatives by measuring how close each one is to the **positive ideal solution** and how far it is from the **negative ideal solution**.

For each alternative $i$:

$$
R_i = \frac{d_i^-}{d_i^+ + d_i^-}
$$

where:

- $d_i^+$ is the distance from alternative $i$ to the positive ideal solution
- $d_i^-$ is the distance from alternative $i$ to the negative ideal solution
- a **larger** $R_i$ means a better alternative

## Step 1: Normalize the decision matrix

If $x_{ij}$ is the value of alternative $i$ on criterion $j$, the normalized value is:

$$
r_{ij} = \frac{x_{ij}}{\sqrt{\sum_{i=1}^{m} x_{ij}^2}}
$$

This makes every criterion column comparable even if the original units are different.

## Step 2: Apply criterion weights

After normalization, each column is multiplied by its weight $w_j$:

$$
v_{ij} = w_j \cdot r_{ij}
$$

This gives the weighted normalized decision matrix.

## Step 3: Find the ideal solutions

For a **benefit criterion** (larger is better):

$$
v_j^+ = \max_i(v_{ij}), \qquad v_j^- = \min_i(v_{ij})
$$

For a **cost criterion** (smaller is better):

$$
v_j^+ = \min_i(v_{ij}), \qquad v_j^- = \max_i(v_{ij})
$$

## Step 4: Compute distances

The distance to the positive ideal solution is:

$$
d_i^+ = \sqrt{\sum_{j=1}^{n}(v_{ij} - v_j^+)^2}
$$

The distance to the negative ideal solution is:

$$
d_i^- = \sqrt{\sum_{j=1}^{n}(v_{ij} - v_j^-)^2}
$$

## Step 5: Compute final ranking score

The relative closeness is:

$$
R_i = \frac{d_i^-}{d_i^+ + d_i^-}
$$

The closer $R_i$ is to 1, the better the alternative.

## Example

The tutorial below is split into small code parts: first we prepare the input data, then we run TOPSIS, and finally we interpret the ranking.

The notebook imports the implementation from [TOPSIS.py](TOPSIS.py), so you only need to keep the notebook and the module in the same folder.

In [2]:
import sys
import os
import numpy as np

# Adds the parent directory to sys.path
sys.path.insert(0, os.path.abspath('..'))

# Now you can import your package
from TOPSIS.models.TOPSIS import TOPSIS

# Example decision matrix:
# Rows = alternatives
# Columns = criteria
# Here, the third criterion is a cost criterion, so lower values are better.
decision_matrix = [
    [7, 9, 9],
    [8, 7, 8],
    [6, 8, 7]
]

# Criterion weights should reflect how important each criterion is.
# In a typical TOPSIS setup, they are normalized to sum to 1.
weight_vector = [0.3, 0.4, 0.3]

# True  = benefit criterion (higher is better)
# False = cost criterion (lower is better)
flag = [True, True, False]

## Part 2: Run TOPSIS

Now we create the model, calculate the TOPSIS scores, and rank the alternatives.

In [3]:
model = TOPSIS(decision_matrix, weight_vector, flag)

print("Decision matrix:\n", model.decision_matrix)
print("\nStandardized matrix:\n", model.standardized_matrix)
print("\nWeighted standardized matrix:\n", model.weighted_standardized_matrix)
print("\nPositive ideal solution components:\n", model.positive_ideal_solution)
print("\nNegative ideal solution components:\n", model.negative_ideal_solution)
print("\nRelative closeness scores:\n", model.relative_closeness)

# Rank alternatives from best to worst.
ranking = np.argsort(model.relative_closeness)[::-1]
print("\nRanking order (best to worst):", ranking)

Decision matrix:
 [[7. 9. 9.]
 [8. 7. 8.]
 [6. 8. 7.]]

Standardized matrix:
 [[0.57346234 0.64616234 0.64616234]
 [0.65538554 0.50257071 0.57436653]
 [0.49153915 0.57436653 0.50257071]]

Weighted standardized matrix:
 [[0.1720387  0.25846494 0.1938487 ]
 [0.19661566 0.20102828 0.17230996]
 [0.14746175 0.22974661 0.15077121]]

Positive ideal solution components:
 [[0.00060403 0.         0.00185567]
 [0.         0.00329897 0.00046392]
 [0.00241611 0.00082474 0.        ]]

Negative ideal solution components:
 [[0.00060403 0.00329897 0.        ]
 [0.00241611 0.         0.00046392]
 [0.         0.00082474 0.00185567]]

Relative closeness scores:
 [0.55745833 0.46662627 0.4762847 ]

Ranking order (best to worst): [0 2 1]


## Part 3: Read the results

The final score is the `relative_closeness` array. A larger score means the alternative is closer to the ideal solution and farther from the worst solution.

If the ranking order prints something like `[1 0 2]`, it means:

1. Alternative 2 is the best choice
2. Alternative 1 is second
3. Alternative 3 is third